In [1]:
!pip install -q sentence-transformers faiss-cpu langchain-text-splitters transformers accelerate

In [2]:
from langchain_core.documents import Document

raw = [
 # id, text, source, category, document_type, date
 (0,"Refund policy (2021): Customers may request a refund within 14 days of purchase on any plan.","policy_2021.pdf","pricing","policy","2021-03-01"),
 (1,"Refund policy (2024): Customers may request a refund within 30 days of purchase. Annual plans are refunded pro-rata after 30 days.","policy_2024.pdf","pricing","policy","2024-02-10"),
 (2,"Veridia Labs offers flexible plans with generous refund guarantees so you can try risk-free.","website.html","pricing","marketing","2024-05-01"),
 (3,"Pricing: Starter costs $19 per month, Pro costs $49 per month, Enterprise is custom.","pricing_faq.md","pricing","faq","2023-06-15"),
 (4,"Pricing (2020): Starter costs $9 per month, Pro costs $29 per month.","pricing_2020.md","pricing","policy","2020-01-20"),
 (5,"Veridia Vault encrypts all data at rest with AES-256 and in transit with TLS 1.3.","security_ref_2024.pdf","security","technical_reference","2024-03-12"),
 (6,"Data is encrypted at rest with AES-128 and in transit with TLS 1.2.","security_ref_2020.pdf","security","technical_reference","2020-08-05"),
 (7,"Bank-grade security keeps your data safe. Veridia Labs takes privacy and encryption seriously.","website.html","security","marketing","2024-05-01"),
 (8,"Data retention: deleted customer data is permanently purged from backups within 90 days.","retention_policy.pdf","security","policy","2023-09-01"),
 (9,"Support hours: Monday to Friday, 9am to 6pm IST. Enterprise customers get 24/7 phone support.","support_faq_2024.md","support","faq","2024-01-08"),
 (10,"Support hours: Monday to Friday, 10am to 5pm IST. Email support only.","support_faq_2021.md","support","faq","2021-04-02"),
 (11,"Our friendly support team is always ready to help you succeed with Veridia Labs.","website.html","support","marketing","2024-05-01"),
 (12,"Veridia API rate limit: 1000 requests per minute on Pro, 5000 on Enterprise. Exceeding the limit returns HTTP 429.","api_ref_2024.pdf","product","technical_reference","2024-06-20"),
 (13,"Veridia API rate limit: 100 requests per minute for all plans.","api_ref_2021.pdf","product","technical_reference","2021-07-11"),
 (14,"Veridia Labs APIs are fast, scalable and developer friendly.","website.html","product","marketing","2024-05-01"),
 (15,"Release 4.2 (2025): added single sign-on via SAML and a dark mode dashboard.","release_4_2.md","product","release_notes","2025-02-14"),
 (16,"Release 3.0 (2022): launched the analytics dashboard and CSV export.","release_3_0.md","product","release_notes","2022-05-30"),
 (17,"Remote work policy: employees may work remotely up to 3 days per week.","hr_policy_2024.pdf","hr","policy","2024-04-01"),
 (18,"Remote work policy: employees must work from the Anantnag office 5 days per week.","hr_policy_2021.pdf","hr","policy","2021-02-15"),
 (19,"Join Veridia Labs, a great place to work with an amazing culture.","careers.html","hr","marketing","2024-05-01"),
 # --- two deliberately awkward docs for the edge cases ---
 (20,"Refunds are sent back to the original card and take 5 business days to arrive.","billing_notes.txt","pricing","faq",None),   # MISSING date
 (21,"Billing security: card details are tokenised by our payment processor and never stored on Veridia servers.","billing_security.pdf","security","policy","2024-03-01"),  # belongs to BOTH pricing and security
]
docs = []
for i,text,src,cat,dtype,date in raw:
    meta = {"doc_id":i,"source":src,"category":cat,"document_type":dtype}
    if date: meta["date"] = date          # doc 20 intentionally has no date key
    docs.append(Document(page_content=text, metadata=meta))
print(len(docs), "documents;", sum('date' not in d.metadata for d in docs), "with missing date")

22 documents; 1 with missing date


In [3]:
DOCUMENTS = []
for i, text, src, cat, dtype, date in raw:
    DOCUMENTS.append({
        "id": f"doc{i}",
        "title": src,            # e.g. policy_2024.pdf, shown as the citation
        "category": cat,
        "document_type": dtype,
        "date": date,            # doc 20 will be None, that's the edge case
        "text": text,
    })

print("Number of documents:", len(DOCUMENTS))
print(DOCUMENTS[1])

Number of documents: 22
{'id': 'doc1', 'title': 'policy_2024.pdf', 'category': 'pricing', 'document_type': 'policy', 'date': '2024-02-10', 'text': 'Refund policy (2024): Customers may request a refund within 30 days of purchase. Annual plans are refunded pro-rata after 30 days.'}


In [4]:
import faiss
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

embedder = SentenceTransformer("all-MiniLM-L6-v2")
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

chunks = []
for doc in DOCUMENTS:
    for i, piece in enumerate(splitter.split_text(doc["text"])):
        chunks.append({
            "chunk_id": f'{doc["id"]}-{i}',
            "title": doc["title"],
            "category": doc["category"],
            "document_type": doc["document_type"],
            "date": doc["date"],
            "text": piece,
        })

vectors = embedder.encode([c["text"] for c in chunks],
                          normalize_embeddings=True).astype("float32")
index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)

print("Number of chunks:", len(chunks))
print("First chunk:", chunks[0])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Number of chunks: 22
First chunk: {'chunk_id': 'doc0-0', 'title': 'policy_2021.pdf', 'category': 'pricing', 'document_type': 'policy', 'date': '2021-03-01', 'text': 'Refund policy (2021): Customers may request a refund within 14 days of purchase on any plan.'}


In [5]:
def retrieve(query, top_k=3, category=None, document_type=None, min_date=None):
    # 1. Turn the question into numbers (same model as the chunks)
    q = embedder.encode([query], normalize_embeddings=True).astype("float32")

    # 2. Search ALL chunks, so filtering can't leave us with too few results
    scores, idxs = index.search(q, len(chunks))

    # 3. Walk through results from best to worst, keep only those that pass the filters
    results = []
    for score, i in zip(scores[0], idxs[0]):
        c = chunks[i]
        if category and c["category"] != category:
            continue
        if document_type and c["document_type"] != document_type:
            continue
        if min_date and (c["date"] is None or c["date"] < min_date):
            continue
        results.append({**c, "score": float(score)})
        if len(results) == top_k:
            break
    return results

In [6]:
for q in ["How long do I have to ask for a refund?",
          "What is the API rate limit on the Pro plan?",
          "What is the capital of France?"]:
    print("\nQUESTION:", q)
    for r in retrieve(q):
        print(f'  {r["score"]:.3f} | {r["chunk_id"]} | {r["title"]} | {r["text"][:60]}')

print("\n--- With filter: category=security, dates from 2024 onward ---")
for r in retrieve("How is my data encrypted?", category="security", min_date="2024-01-01"):
    print(f'  {r["score"]:.3f} | {r["chunk_id"]} | {r["title"]} | {r["date"]}')


QUESTION: How long do I have to ask for a refund?
  0.715 | doc0-0 | policy_2021.pdf | Refund policy (2021): Customers may request a refund within 
  0.590 | doc1-0 | policy_2024.pdf | Refund policy (2024): Customers may request a refund within 
  0.564 | doc20-0 | billing_notes.txt | Refunds are sent back to the original card and take 5 busine

QUESTION: What is the API rate limit on the Pro plan?
  0.601 | doc13-0 | api_ref_2021.pdf | Veridia API rate limit: 100 requests per minute for all plan
  0.597 | doc12-0 | api_ref_2024.pdf | Veridia API rate limit: 1000 requests per minute on Pro, 500
  0.461 | doc4-0 | pricing_2020.md | Pricing (2020): Starter costs $9 per month, Pro costs $29 pe

QUESTION: What is the capital of France?
  0.072 | doc19-0 | careers.html | Join Veridia Labs, a great place to work with an amazing cul
  0.062 | doc7-0 | website.html | Bank-grade security keeps your data safe. Veridia Labs takes
  0.059 | doc13-0 | api_ref_2021.pdf | Veridia API rate limit: 100

In [7]:
import torch
from transformers import pipeline

llm = pipeline("text-generation",
               model="Qwen/Qwen2.5-1.5B-Instruct",
               torch_dtype=torch.float16,
               device_map="auto")
print("Model loaded")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model loaded


In [8]:
CONFIDENCE_THRESHOLD = 0.3

GROUNDING_PROMPT = """You are a knowledge assistant for Veridia Labs.
Answer ONLY using the context below. Never use outside knowledge.
Each context item shows [chunk_id | source file | date].
If sources conflict, trust the one with the most recent date.
If the context does not contain the answer, reply exactly:
"I don't have enough information in my knowledge base to answer that."
Mention the chunk_id(s) you used in your answer.

Context:
{context}

Question: {question}
Answer:"""


def ask(query, top_k=3, **filters):
    hits = retrieve(query, top_k=top_k, **filters)
    top_score = max((h["score"] for h in hits), default=0.0)

    # newest source first, so the model sees the current info before the old
    ordered = sorted(hits, key=lambda h: h["date"] or "", reverse=True)
    context = "\n".join(
        f'[{h["chunk_id"]} | {h["title"]} | {h["date"]}] {h["text"]}'
        for h in ordered)

    prompt = GROUNDING_PROMPT.format(context=context, question=query)
    out = llm([{"role": "user", "content": prompt}],
              max_new_tokens=150, do_sample=False)
    answer = out[0]["generated_text"][-1]["content"].strip()

    low_conf = top_score < CONFIDENCE_THRESHOLD
    if low_conf:
        answer += ("\n\n⚠️ Low confidence: the best retrieval match was weak "
                   "(score below 0.3), so this answer may not be well-supported.")

    return {
        "answer": answer,
        "sources": [{"chunk_id": h["chunk_id"], "title": h["title"],
                     "score": round(h["score"], 3)} for h in hits],
        "top_score": round(top_score, 3),
        "low_confidence": low_conf,
    }

In [9]:
import json
for q in ["How long do I have to ask for a refund?",
          "What is the API rate limit on the Pro plan?",
          "What is the capital of France?"]:
    print("\nQUESTION:", q)
    print(json.dumps(ask(q), indent=2, ensure_ascii=False))


QUESTION: How long do I have to ask for a refund?
{
  "answer": "I don't have enough information in my knowledge base to answer that.",
  "sources": [
    {
      "chunk_id": "doc0-0",
      "title": "policy_2021.pdf",
      "score": 0.715
    },
    {
      "chunk_id": "doc1-0",
      "title": "policy_2024.pdf",
      "score": 0.59
    },
    {
      "chunk_id": "doc20-0",
      "title": "billing_notes.txt",
      "score": 0.564
    }
  ],
  "top_score": 0.715,
  "low_confidence": false
}

QUESTION: What is the API rate limit on the Pro plan?
{
  "answer": "The API rate limit on the Pro plan is 1000 requests per minute.",
  "sources": [
    {
      "chunk_id": "doc13-0",
      "title": "api_ref_2021.pdf",
      "score": 0.601
    },
    {
      "chunk_id": "doc12-0",
      "title": "api_ref_2024.pdf",
      "score": 0.597
    },
    {
      "chunk_id": "doc4-0",
      "title": "pricing_2020.md",
      "score": 0.461
    }
  ],
  "top_score": 0.601,
  "low_confidence": false
}

QUESTI

In [10]:
REFUSAL = "I don't have enough information in my knowledge base to answer that."

GROUNDING_PROMPT = """You are a knowledge assistant for Veridia Labs.
Answer ONLY using the context below. Never use outside knowledge.
Each context item shows [chunk_id | source file | date].

Rules:
1. If the context contains information that answers the question, answer it in one or two sentences.
2. If two items give different values, they are old and new versions. Use the item with the most recent date, and say the older value is outdated.
3. Only if NO context item relates to the question, reply exactly: "{refusal}"

Context:
{context}

Question: {question}
Answer:"""


def ask(query, top_k=3, **filters):
    hits = retrieve(query, top_k=top_k, **filters)
    top_score = max((h["score"] for h in hits), default=0.0)

    ordered = sorted(hits, key=lambda h: h["date"] or "", reverse=True)
    context = "\n".join(
        f'[{h["chunk_id"]} | {h["title"]} | {h["date"]}] {h["text"]}'
        for h in ordered)

    prompt = GROUNDING_PROMPT.format(refusal=REFUSAL, context=context, question=query)
    out = llm([{"role": "user", "content": prompt}],
              max_new_tokens=150, do_sample=False)
    answer = out[0]["generated_text"][-1]["content"].strip()

    # citation added by code, only when the model actually answered
    refused = answer.startswith(REFUSAL[:30])
    if not refused:
        cites = ", ".join(f'{h["chunk_id"]} ({h["title"]})' for h in hits)
        answer += f"\n\nSources: {cites}"

    low_conf = top_score < CONFIDENCE_THRESHOLD
    if low_conf:
        answer += ("\n\n⚠️ Low confidence: the best retrieval match was weak "
                   "(score below 0.3), so this answer may not be well-supported.")

    return {
        "answer": answer,
        "sources": [{"chunk_id": h["chunk_id"], "title": h["title"],
                     "score": round(h["score"], 3)} for h in hits],
        "top_score": round(top_score, 3),
        "low_confidence": low_conf,
    }

In [11]:
for q in ["How long do I have to ask for a refund?",
          "What are the support hours?",
          "How many days can employees work remotely?",
          "What is the capital of France?"]:
    r = ask(q)
    print("\nQUESTION:", q)
    print(r["answer"])
    print("top_score:", r["top_score"], "| low_confidence:", r["low_confidence"])


QUESTION: How long do I have to ask for a refund?
According to the refund policy from 2021, customers can request a refund within 14 days of purchase on any plan. The newer policy from 2024 allows refunds up to 30 days after purchase, but only applies to annual plans.

Sources: doc0-0 (policy_2021.pdf), doc1-0 (policy_2024.pdf), doc20-0 (billing_notes.txt)
top_score: 0.715 | low_confidence: False

QUESTION: What are the support hours?
The support hours are as follows:

- For enterprise customers: 24/7 phone support available.
- For regular customers: Monday to Friday, from 10am to 5pm IST via email.

Sources: doc9-0 (support_faq_2024.md), doc10-0 (support_faq_2021.md), doc17-0 (hr_policy_2024.pdf)
top_score: 0.699 | low_confidence: False

QUESTION: How many days can employees work remotely?
Employees can work remotely up to 3 days per week according to the remote work policy as of April 1, 2024.

Sources: doc17-0 (hr_policy_2024.pdf), doc18-0 (hr_policy_2021.pdf), doc9-0 (support_faq_

In [12]:
import re

def newest_versions_only(hits):
    # policy_2021.pdf and policy_2024.pdf -> same base name "policy.pdf"
    base = lambda t: re.sub(r"_(19|20)\d{2}(?=\.)", "", t)
    best = {}
    for h in hits:
        k = base(h["title"])
        if k not in best or (h["date"] or "") > (best[k]["date"] or ""):
            best[k] = h
    return [h for h in hits if best[base(h["title"])] is h]


def ask(query, top_k=3, **filters):
    # fetch extra, drop outdated versions, then keep the best top_k
    hits = retrieve(query, top_k=6, **filters)
    hits = newest_versions_only(hits)[:top_k]
    top_score = max((h["score"] for h in hits), default=0.0)
    low_conf = top_score < CONFIDENCE_THRESHOLD

    if low_conf:
        # gate: don't let the model answer from its own knowledge
        answer = REFUSAL
    else:
        ordered = sorted(hits, key=lambda h: h["date"] or "", reverse=True)
        context = "\n".join(
            f'[{h["chunk_id"]} | {h["title"]} | {h["date"]}] {h["text"]}'
            for h in ordered)
        prompt = GROUNDING_PROMPT.format(refusal=REFUSAL, context=context, question=query)
        out = llm([{"role": "user", "content": prompt}],
                  max_new_tokens=150, do_sample=False)
        answer = out[0]["generated_text"][-1]["content"].strip()
        if not answer.startswith(REFUSAL[:30]):
            cites = ", ".join(f'{h["chunk_id"]} ({h["title"]})' for h in hits)
            answer += f"\n\nSources: {cites}"

    if low_conf:
        answer += ("\n\n⚠️ Low confidence: the best retrieval match was weak "
                   "(score below 0.3), so this answer may not be well-supported.")

    return {
        "answer": answer,
        "sources": [{"chunk_id": h["chunk_id"], "title": h["title"],
                     "score": round(h["score"], 3)} for h in hits],
        "top_score": round(top_score, 3),
        "low_confidence": low_conf,
    }

In [13]:
for q in ["How long do I have to ask for a refund?",
          "What are the support hours?",
          "How many days can employees work remotely?",
          "What is the API rate limit on the Pro plan?",
          "What is the capital of France?"]:
    r = ask(q)
    print("\nQUESTION:", q)
    print(r["answer"])
    print("top_score:", r["top_score"], "| low_confidence:", r["low_confidence"])


QUESTION: How long do I have to ask for a refund?
According to the refund policy from 2024, customers have up to 30 days from the purchase date to request a refund.

Sources: doc1-0 (policy_2024.pdf), doc20-0 (billing_notes.txt), doc2-0 (website.html)
top_score: 0.59 | low_confidence: False

QUESTION: What are the support hours?
The support hours are Monday to Friday from 9am to 6pm IST. Enterprise customers also receive 24/7 phone support.

Sources: doc9-0 (support_faq_2024.md), doc17-0 (hr_policy_2024.pdf), doc13-0 (api_ref_2021.pdf)
top_score: 0.699 | low_confidence: False


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



QUESTION: How many days can employees work remotely?
Employees may work remotely up to 3 days per week.

Sources: doc17-0 (hr_policy_2024.pdf), doc9-0 (support_faq_2024.md), doc8-0 (retention_policy.pdf)
top_score: 0.833 | low_confidence: False

QUESTION: What is the API rate limit on the Pro plan?
The API rate limit on the Pro plan is 1000 requests per minute.

Sources: doc12-0 (api_ref_2024.pdf), doc4-0 (pricing_2020.md), doc3-0 (pricing_faq.md)
top_score: 0.597 | low_confidence: False

QUESTION: What is the capital of France?
I don't have enough information in my knowledge base to answer that.

⚠️ Low confidence: the best retrieval match was weak (score below 0.3), so this answer may not be well-supported.
top_score: 0.072 | low_confidence: True


In [14]:
!pip install -q fastapi uvicorn

In [15]:
import threading, time, uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from typing import Optional

app = FastAPI(title="Veridia Knowledge Assistant")

class AskRequest(BaseModel):
    query: str
    category: Optional[str] = None

@app.post("/ask")
def ask_endpoint(req: AskRequest):
    filters = {"category": req.category} if req.category else {}
    return ask(req.query, **filters)

server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)
print("Server is running on http://127.0.0.1:8000")

Server is running on http://127.0.0.1:8000


In [16]:
print("--- TEST 1: normal question ---")
!curl -s -X POST http://127.0.0.1:8000/ask -H "Content-Type: application/json" -d '{"query": "How long do I have to ask for a refund?"}' | python -m json.tool

print("\n--- TEST 2: out-of-scope question ---")
!curl -s -X POST http://127.0.0.1:8000/ask -H "Content-Type: application/json" -d '{"query": "What is the capital of France?"}' | python -m json.tool

print("\n--- TEST 3: bad request (no query field) ---")
!curl -s -X POST http://127.0.0.1:8000/ask -H "Content-Type: application/json" -d '{"question": "hello"}'

--- TEST 1: normal question ---
{
    "answer": "According to the refund policy from 2024, customers have up to 30 days from the purchase date to request a refund.\n\nSources: doc1-0 (policy_2024.pdf), doc20-0 (billing_notes.txt), doc2-0 (website.html)",
    "sources": [
        {
            "chunk_id": "doc1-0",
            "title": "policy_2024.pdf",
            "score": 0.59
        },
        {
            "chunk_id": "doc20-0",
            "title": "billing_notes.txt",
            "score": 0.564
        },
        {
            "chunk_id": "doc2-0",
            "title": "website.html",
            "score": 0.338
        }
    ],
    "top_score": 0.59,
    "low_confidence": false
}

--- TEST 2: out-of-scope question ---
{
    "answer": "I don't have enough information in my knowledge base to answer that.\n\n\u26a0\ufe0f Low confidence: the best retrieval match was weak (score below 0.3), so this answer may not be well-supported.",
    "sources": [
        {
            "chunk_id":

In [17]:
import requests, json
import pandas as pd

REFUSAL_START = "I don't have enough information"

# (difficulty, question, expected chunk ids, expected facts; "|" separates alternatives)
TESTS = [
    ("easy",   "How long do I have to ask for a refund?",                 ["doc1-0"],  ["30"]),
    ("easy",   "How many days can employees work remotely?",              ["doc17-0"], ["3"]),
    ("easy",   "What is the API rate limit on the Pro plan?",             ["doc12-0"], ["1000"]),
    ("easy",   "What encryption does Veridia Vault use for data at rest?",["doc5-0"],  ["AES-256"]),
    ("easy",   "How long are deleted customer records kept in backups?",  ["doc8-0"],  ["90"]),

    ("medium", "When can I reach support by phone?",                      ["doc9-0"],  ["24/7"]),
    ("medium", "What happens if I exceed the API rate limit?",            ["doc12-0"], ["429"]),
    ("medium", "How long does it take for a refund to reach my card?",    ["doc20-0"], ["5 business|five business"]),
    ("medium", "Which release added single sign-on?",                     ["doc15-0"], ["4.2|SAML"]),
    ("medium", "Are my card details stored on Veridia's servers?",        ["doc21-0"], ["tokeni|never stored|not stored"]),

    ("hard",   "How much does the Pro plan cost?",                        ["doc3-0"],  ["49"]),
    ("hard",   "What TLS version is used for data in transit?",           ["doc5-0"],  ["1.3"]),
    ("hard",   "Do annual plans get refunded differently from monthly plans?", ["doc1-0"], ["pro-rata|pro rata|prorata"]),
    ("hard",   "What is the annual revenue of Veridia Labs?",             [],          []),   # out of scope
    ("hard",   "What is the boiling point of water?",                     [],          []),   # out of scope
]

rows = []
for difficulty, question, expected_ids, facts in TESTS:
    r = requests.post("http://127.0.0.1:8000/ask", json={"query": question}).json()
    answer = r["answer"].split("\n\nSources:")[0].split("\n\n⚠️")[0].strip()
    retrieved = [s["chunk_id"] for s in r["sources"]]
    refused = answer.startswith(REFUSAL_START)

    if expected_ids:   # in-scope question
        ranks = [retrieved.index(c) + 1 for c in expected_ids if c in retrieved]
        retrieval_score = {1: 5, 2: 4, 3: 3}.get(min(ranks), 1) if ranks else 1
        hit = sum(any(alt.lower() in answer.lower() for alt in f.split("|")) for f in facts)
        response_score = 1 if refused else (5 if hit == len(facts) else 3 if hit else 1)
    else:              # out-of-scope: the correct behaviour is to refuse
        retrieval_score = 5 if r["low_confidence"] else 1
        response_score = 5 if refused else 1

    rows.append({"difficulty": difficulty, "question": question,
                 "top_score": r["top_score"], "low_conf": r["low_confidence"],
                 "retrieved": retrieved, "retrieval_score": retrieval_score,
                 "response_score": response_score, "answer": answer})

df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 70, "display.width", 250)
print(df[["difficulty", "question", "top_score", "retrieval_score", "response_score"]])
print("\nAVERAGES BY DIFFICULTY")
print(df.groupby("difficulty")[["retrieval_score", "response_score"]].mean().round(2))
print("\nOVERALL:", df[["retrieval_score", "response_score"]].mean().round(2).to_dict())

df.to_csv("test_results.csv", index=False)
json.dump(rows, open("test_results.json", "w"), indent=2, ensure_ascii=False)

   difficulty                                                      question  top_score  retrieval_score  response_score
0        easy                       How long do I have to ask for a refund?      0.590                5               5
1        easy                    How many days can employees work remotely?      0.833                5               5
2        easy                   What is the API rate limit on the Pro plan?      0.597                5               5
3        easy      What encryption does Veridia Vault use for data at rest?      0.800                5               5
4        easy        How long are deleted customer records kept in backups?      0.771                5               5
5      medium                            When can I reach support by phone?      0.662                5               5
6      medium                  What happens if I exceed the API rate limit?      0.501                5               5
7      medium          How long does it 

In [18]:
for _, row in df.iterrows():
    print(f'\n[{row.difficulty}] {row.question}')
    print(f'   retrieval {row.retrieval_score} | response {row.response_score} | top_score {row.top_score}')
    print('  ', row.answer)


[easy] How long do I have to ask for a refund?
   retrieval 5 | response 5 | top_score 0.59
   According to the refund policy from 2024, customers have up to 30 days from the purchase date to request a refund.

[easy] How many days can employees work remotely?
   retrieval 5 | response 5 | top_score 0.833
   Employees may work remotely up to 3 days per week.

[easy] What is the API rate limit on the Pro plan?
   retrieval 5 | response 5 | top_score 0.597
   The API rate limit on the Pro plan is 1000 requests per minute.

[easy] What encryption does Veridia Vault use for data at rest?
   retrieval 5 | response 5 | top_score 0.8
   Veridia Vault uses AES-256 encryption for data at rest.

[easy] How long are deleted customer records kept in backups?
   retrieval 5 | response 5 | top_score 0.771
   Deleted customer records are permanently purged from backups within 90 days.

[medium] When can I reach support by phone?
   retrieval 5 | response 5 | top_score 0.662
   Enterprise customers c

In [19]:
df.loc[df.question.str.startswith("Do annual plans"), "response_score"] = 4
df.to_csv("test_results.csv", index=False)
json.dump(df.to_dict("records"), open("test_results.json", "w"), indent=2, ensure_ascii=False)

print(df.groupby("difficulty")[["retrieval_score", "response_score"]].mean().round(2))
print("OVERALL:", df[["retrieval_score", "response_score"]].mean().round(2).to_dict())

            retrieval_score  response_score
difficulty                                 
easy                    5.0             5.0
hard                    3.8             4.8
medium                  5.0             5.0
OVERALL: {'retrieval_score': 4.6, 'response_score': 4.93}
